In [1]:
!nvidia-smi

Sun Apr 28 11:22:37 2024       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 470.103.01   Driver Version: 470.103.01   CUDA Version: 11.4     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla K80           Off  | 00000000:05:00.0 Off |                    0 |
| N/A   42C    P0    57W / 149W |   6937MiB / 11441MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  Tesla K80           Off  | 00000000:06:00.0 Off |                    0 |
| N/A   

In [2]:
import os
import nibabel as nib
import numpy as np 
from collections import OrderedDict
import json
from pathlib import Path

from utils.helper import convert_nrrd_to_nifti, create_folder, plot_all_slices, plot_histogram, generate_binary_image, adjust_affine_for_spacing_and_origin, save_binary_image_with_adjusted_origin, make_if_dont_exist
from utils.metrics import dice_score_per_class, hausdorff_distance_per_class, ravd_per_class

In [3]:

# define dataset path
BASE_PATH = Path('./').resolve()
DATA_PATH = BASE_PATH / 'dataset'

project_name = 'HCFC1' #change here for different task name
task_name = 'Dataset002_' + project_name 

TRAINING_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'imagesTr'
GT_TRAINING_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'labelsTr'
TEST_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'imagesTs'
GT_TEST_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'labelsTs'
PREDICTION_RESULTS_PATH  = BASE_PATH / 'dataset/nnUNet_Prediction_Results' / task_name
TASK_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name 

# setup environment variables
nnUNet_raw = BASE_PATH / 'dataset/nnUNet_raw_data'
nnUNet_preprocessed = BASE_PATH / 'dataset/nnUNet_preprocessed'
nnUNet_results = BASE_PATH / 'dataset/nnUNet_results'

In [5]:
make_if_dont_exist(TRAINING_DATASET_PATH,overwrite=False)
make_if_dont_exist(GT_TRAINING_DATASET_PATH)
make_if_dont_exist(TEST_DATASET_PATH)
make_if_dont_exist(GT_TEST_DATASET_PATH)
make_if_dont_exist(PREDICTION_RESULTS_PATH)

make_if_dont_exist(nnUNet_preprocessed)
make_if_dont_exist(nnUNet_results)

/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data/Dataset002_HCFC1/imagesTr exists.
/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data/Dataset002_HCFC1/labelsTr exists.
/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data/Dataset002_HCFC1/imagesTs created!
/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data/Dataset002_HCFC1/labelsTs created!
/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_Prediction_Results/Dataset002_HCFC1 created!
/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_preprocessed exists.
/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_results exists.


In [4]:
train_files = os.listdir(TRAINING_DATASET_PATH)
label_files = os.listdir(GT_TRAINING_DATASET_PATH)
print("train image files:",len(train_files))
print("train label files:",len(label_files))
print("Matches:",len(set(train_files).intersection(set(label_files))))


train image files: 12
train label files: 12
Matches: 0


In [5]:
%env nnUNet_raw=$nnUNet_raw
%env nnUNet_preprocessed=$nnUNet_preprocessed
%env nnUNet_results=$nnUNet_results

env: nnUNet_raw=/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data
env: nnUNet_preprocessed=/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_preprocessed
env: nnUNet_results=/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_results


In [7]:
!nnUNetv2_plan_and_preprocess -d 2 --verify_dataset_integrity

Fingerprint extraction...
Dataset002_HCFC1
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Experiment planning...
Attempting to find 3d_lowres config. 
Current spacing: [1.03 1.03 1.03]. 
Current patch size: (112, 224, 80). 
Current median shape: [386.40776699 771.3592233  299.02912621]
Attempting to find 3d_lowres config. 
Current spacing: [1.0609 1.0609 1.0609]. 
Current patch size: (112, 224, 80). 
Current median shape: [375.15317184 748.89244981 290.31954001]
Attempting to find 3d_lowres config. 
Current spacing: [1.092727 1.092727 1.092727]. 
Current patch size: (112, 224, 80). 
Current median shape: [364.22638042 727.08004836 281.86363108]
Attempting to find 3d_lowres config. 
Current spacing: [1.12550881 1.12550881 1.12550881]. 
Current patch size: (112, 224, 80). 
Current median shape: [353.61

In [ ]:
!nnUNetv2_train 2 3d_lowres 0 --npz 

In [ ]:
!nnUNetv2_train 2 3d_lowres 1 --npz 

In [ ]:
!nnUNetv2_train 2 3d_lowres 2 --npz 

In [ ]:
!nnUNetv2_train 2 3d_lowres 3 --npz 

In [ ]:
!nnUNetv2_train 2 3d_lowres 4 --npz 

In [8]:
!nnUNetv2_find_best_configuration 2 -c 3d_lowres 



***All results:***
nnUNetTrainer__nnUNetPlans__3d_lowres: 0.7693137405359662

*Best*: nnUNetTrainer__nnUNetPlans__3d_lowres: 0.7693137405359662

***Determining postprocessing for best model/ensemble***
Removing all but the largest foreground region did not improve results!
Results were improved by removing all but the largest component for 1. Dice before: 0.95582 after: 0.95684
Results were improved by removing all but the largest component for 2. Dice before: 0.82286 after: 0.82289
Results were improved by removing all but the largest component for 3. Dice before: 0.93805 after: 0.93806
Results were improved by removing all but the largest component for 4. Dice before: 0.87571 after: 0.87571
Results were improved by removing all but the largest component for 5. Dice before: 0.89957 after: 0.89959
Results were improved by removing all but the largest component for 6. Dice before: 0.86754 after: 0.86755
Removing all but the largest component for 7 did not improve results! Dice before: 

## background remove. 

In [6]:
!nnUNetv2_predict -d Dataset001_Wdr47Kusss -i /work/shared/ngmm/3Dimage/DL_test/source_backgroundremoval/ -o /work/shared/ngmm/3Dimage/DL_test/destination_backgroundremoval -f 0 1 2 3 4 -tr nnUNetTrainer -c 3d_lowres -p nnUNetPlans


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 16 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 16 cases that I would like to predict

Predicting NG2607_8bits_cropped:
perform_everything_on_device: True
100%|███████████████████████████████████████████| 18/18 [03:43<00:00, 12.43s/it]
sending off prediction to background worker for resampling and export
done with NG2607_8bits_cropped

Predicting NG2608_left:
perform_everything_on_device: True
100%|█████████████████████████████████████████████| 9/9 [01:51<00:00, 12.40s/it]
sending off prediction to background worker 

In [7]:
!nnUNetv2_predict -d Dataset002_HCFC1 -i /work/shared/ngmm/3Dimage/DL_test/destination_backgroundremoval -o /work/shared/ngmm/3Dimage/DL_test/target_seg_pred -f 0 1 2 3 4 -tr nnUNetTrainer -c 3d_lowres -p nnUNetPlans


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 16 cases in the source folder
Traceback (most recent call last):
  File "/user1/ngmm/tr855969/.conda/envs/env_tf/bin/nnUNetv2_predict", line 8, in <module>
    sys.exit(predict_entry_point())
  File "/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/nnUNet/nnunetv2/inference/predict_from_raw_data.py", line 850, in predict_entry_point
    predictor.predict_from_files(args.i, args.o, save_probabilities=args.save_probabilities,
  File "/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/nnUNet/nnunetv2/inference/predict_from_raw_data.py", line 245, in predict_from_files

In [ ]:
!nnUNetv2_apply_postprocessing -i /work/shared/ngmm/3Dimage/DL_test/destination_backgroundremoval/predic -o /work/shared/ngmm/3Dimage/DL_test/destination_backgroundremoval/ -pp_pkl_file /beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_results/Dataset002_HCFC1/nnUNetTrainer__nnUNetPlans__3d_lowres/crossval_results_folds_0_1_2_3_4/postprocessing.pkl -np 8 -plans_json /beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_results/Dataset002_HCFC1/nnUNetTrainer__nnUNetPlans__3d_lowres/crossval_results_folds_0_1_2_3_4/plans.json

In [7]:
!nnUNetv2_predict -d Dataset002_HCFC1 -i /work/shared/ngmm/3Dimage/DL_test/source_seg_pred/ -o /work/shared/ngmm/3Dimage/DL_test/target_seg_pred -f 0 1 2 3 4 -tr nnUNetTrainer -c 3d_lowres -p nnUNetPlans


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 2 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 2 cases that I would like to predict

Predicting NG2607_left:
perform_everything_on_device: True
100%|███████████████████████████████████████████| 12/12 [01:48<00:00,  9.03s/it]
sending off prediction to background worker for resampling and export
done with NG2607_left

Predicting NG2607_right:
perform_everything_on_device: True
100%|███████████████████████████████████████████| 18/18 [02:42<00:00,  9.05s/it]
sending off prediction to background worker for resampling and 

In [10]:
!nnUNetv2_predict -d Dataset002_HCFC1 -i /user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_test_data -o /user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_Prediction_Results/Dataset002_HCFC1/3d_lowers -f  1 2 3 4 -tr nnUNetTrainer -c 3d_lowres -p nnUNetPlans


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 1 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 1 cases that I would like to predict

Predicting NG4118_RCL5_masked:
perform_everything_on_device: True
100%|███████████████████████████████████████████| 27/27 [04:04<00:00,  9.06s/it]
sending off prediction to background worker for resampling and export
done with NG4118_RCL5_masked


In [11]:
!nnUNetv2_apply_postprocessing -i /user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_Prediction_Results/Dataset002_HCFC1/3d_lowers -o /user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_postprocessed/Dataset002_HCFC1/3d_lowers -pp_pkl_file /beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_results/Dataset002_HCFC1/nnUNetTrainer__nnUNetPlans__3d_lowres/crossval_results_folds_0_1_2_3_4/postprocessing.pkl -np 8 -plans_json /beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_results/Dataset002_HCFC1/nnUNetTrainer__nnUNetPlans__3d_lowres/crossval_results_folds_0_1_2_3_4/plans.json

### Dice score

In [4]:
import json
from tabulate import tabulate

ds_score = []
hd_score = []
havd_score = []

# Specify the path to your JSON file
json_file_path = '/user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data/Dataset002_HCFC1/dataset.json'

# Read the JSON file
with open(json_file_path, 'r') as file:
    data = json.load(file)

labels = data['labels']
head = ['Volume ID'] + list(labels.keys())

ds_score.append(head)
hd_score.append(head)
havd_score.append(head)

imagePath = "/user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_results/Dataset002_HCFC1/nnUNetTrainer__nnUNetPlans__3d_lowres/fold_1/validation"
gtPath = "/user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data/Dataset002_HCFC1/labelsTr"
all_files = os.listdir(imagePath)
for file in all_files:
    if file.endswith(".nii.gz"):
        imgData = nib.load(os.path.join(imagePath, file)).get_fdata()
        gtData = nib.load(os.path.join(gtPath, file)).get_fdata()
        concatenated_array = np.concatenate(([file], dice_score_per_class(imgData,gtData,25)))

        ds_score.append(concatenated_array)
        hd_score.append(hausdorff_distance_per_class(imgData,gtData,25))
        havd_score.append(ravd_per_class(imgData,gtData,25))

AttributeError: 'str' object has no attribute 'append'

In [5]:
print(tabulate(ds_score, tablefmt="grid"))
print(tabulate(hd_score, tablefmt="grid"))
print(tabulate(havd_score, tablefmt="grid"))

+--------------------+------------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+
| Volume ID          | background | CTX+   | cc+    | CPu    | DG     | HP     | RHP    | A      | ig     | fi     | f      | st     | ic     | och    | ac     | fr     | Hb     | TH     | HY     | MB     | P      | MY     | TCB    | V      | OB     |
+--------------------+------------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+
| NG4117_RCL5.nii.gz | 0.9370     | 0.8375 | 0.3717 | 0.8069 | 0.4581 | 0.6856 | 0.6995 | 0.6687 | 0.0000 | 0.4592 | 0.0725 | 0.3131 | 0.2051 | 0.0716 | 0.0000 | 0.0000 | 0.0000 | 0.6975 | 0.6359 | 0.7230 | 0.8363 | 0.8144 | 0.8836 | 0.2136 | 0

In [1]:
# Function to calculate the average for each row
def calculate_row_average(row_data):
    return sum(row_data) / len(row_data)

In [3]:
import numpy as np
import matplotlib.pyplot as plt

def boxplot(dice_data, hd_data, havd_data, tissue_headers):
    num_regions = len(dice_data)
    num_tissues = len(tissue_headers) - 1  # Subtract 1 for excluding background

    fig, axs = plt.subplots(num_regions, 3, figsize=(18, num_regions * 6))

    for region_idx in range(num_regions):
        if region_idx != 0:
            # Extract scores for the current region
            dice_scores = np.array(dice_data[region_idx])[1:].astype(float)
            hd_scores = np.array(hd_data[region_idx])[1:].astype(float)
            havd_scores = np.array(havd_data[region_idx])[1:].astype(float)

            # Define colors for each boxplot group
            colors = ['lightgreen', 'skyblue', 'lightcoral', 'green']

            # Dice Coefficient Scores
            axs[region_idx, 0].boxplot(dice_scores, patch_artist=True, labels=tissue_headers[1:])
            axs[region_idx, 0].set_title('Region {} - Dice Coefficient Scores'.format(region_idx + 1))
            axs[region_idx, 0].set_xlabel('Tissue Type')
            axs[region_idx, 0].set_ylabel('Score')

            # Hausdorff Distance Scores
            axs[region_idx, 1].boxplot(hd_scores, patch_artist=True, labels=tissue_headers[1:])
            axs[region_idx, 1].set_title('Region {} - Hausdorff Distance Scores'.format(region_idx + 1))
            axs[region_idx, 1].set_xlabel('Tissue Type')
            axs[region_idx, 1].set_ylabel('Score')

            # Relative Absolute Volume Difference Scores
            axs[region_idx, 2].boxplot(havd_scores, patch_artist=True, labels=tissue_headers[1:])
            axs[region_idx, 2].set_title('Region {} - Relative Absolute Volume Difference Scores'.format(region_idx + 1))
            axs[region_idx, 2].set_xlabel('Tissue Type')
            axs[region_idx, 2].set_ylabel('Score')

            # Customize boxplot colors
            for bplot in [axs[region_idx, 0], axs[region_idx, 1], axs[region_idx, 2]]:
                for patch, color in zip(bplot['boxes'], colors):
                    patch.set_facecolor(color)

    plt.tight_layout()
    plt.show()

# Example usage
# Assuming dice_data, hd_data, havd_data, and tissue_headers are defined elsewhere
boxplot(ds_score, hd_score, havd_score, labels)


NameError: name 'ds_score' is not defined

In [4]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Load the NIfTI file
nifti_path = '/user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data/Dataset002_HCFC1/imagesTr/NG4108_RCL5_0000.nii.gz'
# org_nifti_path = '/user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data/Dataset003_HPC/imagesTr/NG4115_RCL5_0000.nii.gz'
img = nib.load(nifti_path)
data = img.get_fdata()

# org_img = nib.load(org_nifti_path).get_fdata()

# data = org_img * data
# Adjusted function to display a slice and its histogram
def display_slice_and_histogram(slice_no):
    # Setup the figure and axes for a side-by-side plot: slice and histogram
    fig, axes = plt.subplots(2, 2, figsize=(16, 8))
    
    # Display the slice
    ax = axes[0,0]
    ax.imshow(data[:, :, slice_no])
    ax.axis('off')  # Hide axes ticks
    ax.set_title(f'Slice {slice_no}')
    
    # Display the histogram
    ax = axes[0,1]
    slice_data = data[:, :, slice_no].ravel()
    ax.hist(slice_data, bins=50, color='c', alpha=0.75)
    ax.set_title('Pixel Intensity Distribution')
    ax.grid(True)

    ax = axes[1,0]
    ax.imshow(data[:, slice_no,: ].T)
    ax.axis('off')  # Hide axes ticks
    ax.set_title(f'Slice {slice_no}')

    ax = axes[1,1]
    ax.imshow(data[slice_no, :,: ])
    ax.axis('off')  # Hide axes ticks
    ax.set_title(f'Slice {slice_no}')
    
    plt.tight_layout()
    plt.show()

# Interactive widget for slice selection
slice_slider = widgets.IntSlider(
    min=0, 
    max=data.shape[2] - 1, 
    step=1, 
    value=data.shape[2] // 2, 
    description='Slice'
)

widgets.interactive(display_slice_and_histogram, slice_no=slice_slider)


interactive(children=(IntSlider(value=201, description='Slice', max=401), Output()), _dom_classes=('widget-int…

## image and gt

In [1]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Load the NIfTI file
nifti_path = '/user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data/Dataset002_HCFC1/imagesTr/NG4110_RCL5_0000.nii.gz'
gt_nifti_path = '/user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data/Dataset002_HCFC1/labelsTr/NG4110_RCL5.nii.gz'
img = nib.load(nifti_path)
data = img.get_fdata()

gt_data = nib.load(gt_nifti_path).get_fdata()

nww_data = gt_data * data
# Adjusted function to display a slice and its histogram
def display_slice_and_histogram(slice_no):
    # Setup the figure and axes for a side-by-side plot: slice and histogram
    fig, axes = plt.subplots(2,4, figsize=(16, 8))
    
    # Display the slice
    ax = axes[0,0]
    ax.imshow(data[:, :, slice_no])
    ax.axis('off')  # Hide axes ticks
    ax.set_title(f'Slice {slice_no}')
    
    # Display the histogram
    ax = axes[0,1]
    slice_data = data[:, :, slice_no].ravel()
    ax.hist(slice_data, bins=50, color='c', alpha=0.75)
    ax.set_title('Img: Pixel Intensity Distribution')
    ax.grid(True)

    ax = axes[1,0]
    ax.imshow(data[:, slice_no,: ].T, cmap='gray')
    ax.axis('off')  # Hide axes ticks
    ax.set_title(f'Img: Slice {slice_no}')

    ax = axes[1,1]
    ax.imshow(data[slice_no, :,: ], cmap='gray')
    ax.axis('off')  # Hide axes ticks
    ax.set_title(f'Img: Slice {slice_no}')

    # gt 
    # Display the slice
    ax = axes[0,2]
    ax.imshow(gt_data[:, :, slice_no], cmap='gray')
    ax.axis('off')  # Hide axes ticks
    ax.set_title(f'GT: Slice {slice_no}')
    
    # Display the histogram
    ax = axes[0,3]
    slice_data = gt_data[:, :, slice_no].ravel()
    ax.hist(slice_data, bins=50, color='c', alpha=0.75)
    ax.set_title('Img: Pixel Intensity Distribution')
    ax.grid(True)

    ax = axes[1,2]
    ax.imshow(gt_data[:, slice_no,: ].T, cmap='gray')
    ax.axis('off')  # Hide axes ticks
    ax.set_title(f'GT: Slice {slice_no}')

    ax = axes[1,3]
    ax.imshow(gt_data[slice_no, :,: ], cmap='gray')
    ax.axis('off')  # Hide axes ticks
    ax.set_title(f'GT: Slice {slice_no}')

    # # new generated data
    # # Display the slice
    # ax = axes[2,0]
    # ax.imshow(nww_data[:, :, slice_no], cmap='gray')
    # ax.axis('off')  # Hide axes ticks
    # ax.set_title(f'Slice {slice_no}')
    
    # # Display the histogram
    # ax = axes[2,1]
    # slice_data = nww_data[:, :, slice_no].ravel()
    # ax.hist(slice_data, bins=50, color='c', alpha=0.75)
    # ax.set_title('Img: Pixel Intensity Distribution')
    # ax.grid(True)

    # ax = axes[3,0]
    # ax.imshow(nww_data[:, slice_no,: ].T, cmap='gray')
    # ax.axis('off')  # Hide axes ticks
    # ax.set_title(f'Slice {slice_no}')

    # ax = axes[3,1]
    # ax.imshow(nww_data[slice_no, :,: ], cmap='gray')
    # ax.axis('off')  # Hide axes ticks
    # ax.set_title(f'Slice {slice_no}')
    
    plt.tight_layout()
    plt.show()

# Interactive widget for slice selection
slice_slider = widgets.IntSlider(
    min=0, 
    max=data.shape[2] - 1, 
    step=1, 
    value=data.shape[2] // 2, 
    description='Slice'
)

widgets.interactive(display_slice_and_histogram, slice_no=slice_slider)


interactive(children=(IntSlider(value=228, description='Slice', max=456), Output()), _dom_classes=('widget-int…